In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import time
from skimage.io import imread
from skimage.segmentation import mark_boundaries
from superpixel import SuperpixelExtractor

def test_single_image_all_algorithms(img_path):
    algorithms = ["slic", "felzenszwalb", "quickshift", "watershed", "seeds"]

    # 加载图像并转为 Tensor
    img = imread(img_path)
    if img.shape[-1] > 3:
        img = img[:, :, :3]
    img_tensor = torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).float()  # [1, 3, H, W]

    # 归一化用于 mark_boundaries
    img_for_vis = img / 255.0 if img.max() > 1 else img

    plt.figure(figsize=(18, 10))
    plt.subplot(2, 3, 1)
    plt.imshow(img_for_vis)
    plt.title("Original Image")
    plt.axis('off')

    for i, algo in enumerate(algorithms):
        print(f"Running {algo}...")
        try:
            extractor = SuperpixelExtractor(algo)
            start = time.time()
            _, n_masks, _, assigned = extractor(img_tensor)
            duration = time.time() - start

            segments = assigned[0].numpy()
            vis_image = mark_boundaries(img_for_vis, segments)

            plt.subplot(2, 3, i + 2)
            plt.imshow(vis_image)
            plt.title(f"{algo.upper()} ({n_masks[0]} masks)\nTime: {duration:.2f}s")
            plt.axis('off')
        except Exception as e:
            print(f"[ERROR] {algo}: {e}")

    plt.tight_layout()
    plt.show()

image_path = "/home/liw324/code/Segment/LKSeg/data/LoveDA/Train/Rural/images_png/0.png"  # 替换为你的图像路径
test_single_image_all_algorithms(image_path)

KeyboardInterrupt: 